In [1]:
import jax
import jax.numpy as jnp

# Sanity check: jax.grad of f(x) = x**2 + 3x should give 2x+3
# At x=2.0, expected: 2*2 + 3 = 7
f = lambda x: x**2 + 3*x
df = jax.grad(f)
print(df(2.0))

7.0


In [4]:
# Kill-check 2: single air/glass interface, normal incidence
# R = ((n1-n2)/(n1+n2))**2
n1 = 1.0  # air
n2 = 1.5  # glass

R = ((n1 - n2) / (n1 + n2))**2
T = 1 - R  # no absorption in glass/air
A = 0.0

print(f"R = {R:.4f}")
print(f"T = {T:.4f}")
print(f"A = {A:.4f}")
print(f"R + T + A = {R + T + A:.4f}")

R = 0.0400
T = 0.9600
A = 0.0000
R + T + A = 1.0000


In [5]:
import jax.numpy as jnp

def fresnel_interface(n1, n2):
    """2x2 interface matrix between materials n1 and n2"""
    return jnp.array([[1,    1   ],
                      [n1,  -n2  ]], dtype=complex)

def propagation_matrix(n, d, lam):
    """2x2 propagation matrix through layer of index n, thickness d, wavelength lam"""
    delta = 2 * jnp.pi * n * d / lam
    return jnp.array([[jnp.exp(1j * delta),  0                  ],
                      [0,                     jnp.exp(-1j * delta)]], dtype=complex)

# Test: at d=0, propagation matrix should be identity
P_test = propagation_matrix(1.5, 0.0, 500.0)
print("P at d=0:")
print(P_test)

P at d=0:
[[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


In [6]:
def tmm_single_layer(n0, n1, n2, d, lam):
    """
    TMM for a single layer stack: n0 / layer(n1, d) / n2
    n0: incident medium (air = 1.0)
    n1: layer refractive index
    n2: substrate (glass = 1.5)
    d: layer thickness in nm
    lam: wavelength in nm
    Returns R, T
    """
    # Build matrices
    D0 = fresnel_interface(n0, n1)   # air -> layer interface
    P1 = propagation_matrix(n1, d, lam)  # through layer
    D1 = fresnel_interface(n1, n2)   # layer -> glass interface
    
    # Total transfer matrix
    D0_inv = jnp.linalg.inv(D0)
    M = D0_inv @ P1 @ D1
    
    # R and T from matrix elements
    t = M[1,1] / M[1,1]  # placeholder - we'll fix this
    r = M[1,0] / M[1,1]
    
    R = jnp.abs(r)**2
    return R

# Kill-check 3: d=0 should give same R as bare Fresnel (0.04)
R_tmm = tmm_single_layer(1.0, 1.5, 1.5, 0.0, 500.0)
print(f"TMM R at d=0: {R_tmm:.4f}")
print(f"Fresnel R:    0.0400")
print(f"Match: {jnp.abs(R_tmm - 0.04) < 1e-6}")

TMM R at d=0: 0.0400
Fresnel R:    0.0400
Match: True
